In [8]:
from dotenv import load_dotenv
from openai import OpenAI
import httpx
import json

load_dotenv()

# verify=False는 SSL 인증서 문제 때문에 임시로 넣은 설정.
client = OpenAI(
    http_client=httpx.Client(verify=False)
)

# 영화 API의 기본 주소
BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"

# messages는 Agent의 메모리 역할을 한다.
# user / assistant / tool 메시지가 계속 쌓이면서 멀티턴 대화가 가능해진다.
messages = [
    {
        "role": "system",
        "content": """
You are a helpful movie expert agent.
Use tools whenever you need real movie data.
Remember previous conversation context, such as movie titles and IDs mentioned earlier.
Always answer in Korean.
"""
    }
]

In [9]:
def get_popular_movies():
    """
    Fetch popular movies from the real movie API.
    """
    response = httpx.get(f"{BASE_URL}/movies", verify=False)
    response.raise_for_status()
    return response.json()


def get_movie_details(id: int):
    """
    Fetch details for a specific movie by movie ID.
    """
    response = httpx.get(f"{BASE_URL}/movies/{id}", verify=False)
    response.raise_for_status()
    return response.json()

def get_similar_movies(id: int):
    """
    Fetch similar movies for a specific movie by movie ID.
    """
    response = httpx.get(f"{BASE_URL}/movies/{id}/similar", verify=False)
    response.raise_for_status()
    return response.json()

# Function Registry
# 모델은 실제 Python 함수를 알지 못하고 함수 이름 문자열만 반환한다.
# 그래서 문자열 이름과 실제 Python 함수를 연결해주는 딕셔너리가 필요하다.
FUNCTIONS = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies,
}

In [10]:
# TOOLS는 모델에게 제공하는 Tool 설명서다.
# 모델은 Python 함수의 내부 코드를 볼 수 없다.
# 대신 이 schema를 보고:
# - 어떤 tool이 있는지
# - 언제 써야 하는지
# - 어떤 argument가 필요한지
# 판단한다.

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get the current popular movies list from the movie API.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a movie using its movie ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID."
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Get similar movie recommendations using a movie ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID."
                    }
                },
                "required": ["id"]
            }
        }
    }
]

In [11]:
def run_tool_calls(tool_calls):
    for tool_call in tool_calls:
        # 모델이 호출하고 싶어 하는 함수 이름
        function_name = tool_call.function.name
        raw_arguments = tool_call.function.arguments

        print(f"\nTool requested: {function_name}")
        print(f"Arguments: {raw_arguments}")
        # JSON 문자열을 Python dict로 변환
        try:
            arguments = json.loads(raw_arguments)
        except json.JSONDecodeError:
            arguments = {}

        # 함수 이름 문자열로 실제 Python 함수 찾기
        function_to_call = FUNCTIONS.get(function_name)
        # 등록되지 않은 함수가 요청된 경우 에러 처리
        if function_to_call is None:
            result = {
                "error": f"Function {function_name} not found."
            }
        else:
            try:
                # **arguments는 dict를 함수 인자로 풀어서 넣는 문법
                # 예: {"id": 693134}
                # → function_to_call(id=693134)
                result = function_to_call(**arguments)
            except Exception as e:
                result = {
                    "error": str(e)
                }
        # Tool 실행 결과를 messages에 저장한다.
        # 이걸 저장해야 다음 LLM 호출 때 모델이 Tool 결과를 읽고 최종 답변을 만들 수 있다.
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result, ensure_ascii=False)
        })

In [12]:
def call_ai():
    while True:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=TOOLS,
            tool_choice="auto"
        )

        assistant_message = response.choices[0].message

        messages.append({
            "role": "assistant",
            "content": assistant_message.content,
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": tool_call.type,
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": tool_call.function.arguments
                    }
                }
                for tool_call in assistant_message.tool_calls
            ] if assistant_message.tool_calls else None
        })

        if assistant_message.tool_calls:
            run_tool_calls(assistant_message.tool_calls)
            continue

        print(f"\nAI: {assistant_message.content}")
        break

In [ ]:
while True:
    user_input = input("\nSend a message to the Movie Agent: ")

    if user_input.lower() in ["exit", "quit"]:
        break

    messages.append({
        "role": "user",
        "content": user_input
    })

    print(f"\nUser: {user_input}")
    call_ai()


User: 지금 인기 있는 영화 알려줘

Tool requested: get_popular_movies
Arguments: {}

AI: 현재 인기 있는 영화 몇 가지를 소개합니다:

1. **Obsession**
   - 개요: 신비로운 "원 원스 윌로우"를 무너뜨리면서 사랑하는 사람의 마음을 얻으려는 한 낙천적인 연인의 이야기.
   - 개봉일: 2026-05-13
   - 평점: 7.9
   - ![포스터](https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg)

2. **Peddi**
   - 개요: 1980년대 안드라 프라데시 시골에서, 한 열정적인 마을 주민이 스포츠를 통해 지역사회를 결집시키는 이야기.
   - 개봉일: 2026-06-03
   - 평점: 6.8
   - ![포스터](https://image.tmdb.org/t/p/w780/kJAJNNBYlbqAcpTDxBNnaILSMTy.jpg)

3. **The Unknown Man**
   - 개요: 플람시 작가가 코트 다쥐르에서 고립되며 영감을 찾으려는 이야기.
   - 개봉일: 2021-10-16
   - 평점: 8.1
   - ![포스터](https://image.tmdb.org/t/p/w780/4TpBhdaSl5ALHbgeaYOLF8Q3haz.jpg)

4. **Michael**
   - 개요: 마이클 잭슨의 삶과 음악을 넘어선 이야기를 담은 영화.
   - 개봉일: 2026-04-22
   - 평점: 8.6
   - ![포스터](https://image.tmdb.org/t/p/w780/zm0KAbOjlt9eR5y7vDiL2dEOwMl.jpg)

5. **Mortal Kombat II**
   - 개요: 팬들이 사랑하는 챔피언들이 다크 룰 Shao Kahn과 싸우는 이야기.
   - 개봉일: 2026-05-06
   - 평점: 8.0
   - ![포스터](https://image.tmdb.org/t/p/w780/hwRdDFI